# narrative_dna Colab Quickstart completo

Este notebook ejecuta `narrative_dna` de punta a punta en Colab: clonado, instalaci?n, segmentaci?n, baseline heur?stico sin LLM, clasificaci?n con OpenAI opcional, adjudicaci?n conservadora, lectura JSON-first, prints legibles, comparación de resultados, CLI y pruebas r?pidas.

Principio del proyecto: el JSON es la fuente de verdad. La notaci?n compacta, por ejemplo `(Y+D)_N0{-}` o `P_N0{0}`, siempre se deriva del JSON validado.

## 1. Clonar el repo desde GitHub

Por defecto se usa la rama `codex/add-llm-timing-logs`. Si ya fue mergeada, cambia `REPO_BRANCH` a `main` o define la variable de entorno `ADNARRATIVA_BRANCH`.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/jcval94/ADNarrativa.git"
REPO_BRANCH = os.environ.get("ADNARRATIVA_BRANCH", "codex/add-llm-timing-logs")
REPO_DIR = Path("/content/ADNarrativa")

if not REPO_DIR.exists():
    print(f"Clonando {REPO_URL} rama={REPO_BRANCH} ...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print(f"Repositorio existente en {REPO_DIR}; actualizando rama={REPO_BRANCH} ...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

os.chdir(REPO_DIR)
branch = subprocess.check_output(["git", "branch", "--show-current"], text=True).strip()
commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip()
print("repo:", Path.cwd())
print("branch:", branch)
print("commit:", commit)

## 2. Instalar dependencias

Instala el paquete en modo editable. En Colab puede tardar un poco la primera vez.

In [ ]:
%pip install -q -e ".[dev]"
print("Instalaci?n completada")

## 3. Imports y sanity check

Este bloque verifica que el paquete local se importe desde `src/` y muestra versiones/configuraci?n clave.

In [ ]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd

REPO_DIR = Path("/content/ADNarrativa")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from narrative_dna.loader import load_text_document  # noqa: E402
from narrative_dna.pipeline import run_pipeline, run_pipeline_from_text  # noqa: E402

llm_config = json.loads(Path("configs/llm_config.json").read_text(encoding="utf-8"))
print("Imports OK")
print("LLM enabled por config:", llm_config["enabled"])
print("Modelo principal:", llm_config["main_classifier"]["model"])
print("Reasoning principal:", llm_config["main_classifier"].get("reasoning_effort"))
print("Text verbosity principal:", llm_config["main_classifier"].get("text_verbosity"))
print("?Temperature configurada en main_classifier?:", "temperature" in llm_config["main_classifier"])

## 4. Textos de ejemplo

El primer texto es deliberadamente exigente: analogía, riesgo, decisión, regla de gobierno, instrucciones y preguntas. El segundo texto es más corto y sirve como comparación.

In [ ]:
transcript_text = "\n".join(
    [
        "Imagina un hospital pequeño que quiere usar IA para priorizar llamadas de pacientes.",
        "El director cree que el sistema debe ser una brújula, no un piloto automático.",
        "En una prueba simulada, diez expedientes con síntomas urgentes suben al inicio",
        "de la cola y tres casos ambiguos quedan marcados para revisión humana.",
        "Pero hay un riesgo serio: si los datos históricos tienen sesgos,",
        "la IA puede repetirlos con apariencia de objetividad.",
        "Por eso la regla de gobierno debe ser simple: la IA recomienda,",
        "el equipo clínico decide y cada cambio queda registrado.",
        "Primero define qu? decisión se automatiza; después mide falsos positivos",
        "y falsos negativos; finalmente publica un reporte mensual de errores.",
        "?A qui?n ayuda este sistema? ?A qui?n podr?a dejar fuera?",
        "?Qu? evidencia necesitar?as antes de confiar en ?l?",
    ]
)

black_box_memory_text = "\n".join(
    [
        "Los aviones tienen una caja negra. Es un registro m?nimo que guarda lo esencial.",
        "Quiz?s podemos hacer algo parecido con nuestra memoria.",
        "Te propongo algo simple: una vez al año, crea tu propia caja negra emocional.",
        "¿Qué aprendiste este año? ¿Qué lograste?",
        "?Qu? quieres dejar registrado para siempre?",
    ]
)

broken_question_text = "Imagina un hospital peque?o con una br?jula operativa. Que aprendiste este año?"

for name, text in {
    "challenging": transcript_text,
    "black_box_memory": black_box_memory_text,
    "broken_question_regression": broken_question_text,
}.items():
    doc = load_text_document(
        text,
        document_id=name,
        source_path="<colab-string>",
        metadata={"source": "colab"},
        language="es",
    )
    print("\n" + "=" * 90)
    print(name)
    print("chars:", len(text), "units:", len(doc.units))
    for unit in doc.units:
        print(f"  {unit.sequence_index:02d} {unit.final_notation:10s} {unit.text}")

## 5. Ejecutar no-LLM baseline

Este modo no llama APIs. Promueve s?lo se?ales heur?sticas determin?sticas de alta confianza, conserva `heuristic_candidates` como evidencia auditable y deja `N_N0{0}` cuando no hay evidencia suficiente.

In [ ]:
no_llm_result = run_pipeline_from_text(
    transcript_text,
    document_id="colab_challenging_demo",
    source_path="<colab-string>",
    metadata={"source": "colab_challenging_example"},
    language="es",
    output_dir="outputs",
    run_id="colab_challenging_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
    log_timings=True,
)

print("run_id:", no_llm_result.run_id)
print("run_dir:", no_llm_result.run_dir)
print("documents:", len(no_llm_result.documents))
print("units:", sum(len(doc.units) for doc in no_llm_result.documents))

## 6. Helpers de inspecci?n con buenos prints

Estas funciones imprimen: ADN compacto, tabla resumen, evidencia, candidatos, flags, relaciones, cadenas y timing. Todo se lee desde objetos/JSON, nunca desde CSV como fuente de verdad.

In [ ]:
def value(x):
    return getattr(x, "value", x)


def values(xs):
    return [value(x) for x in xs]


def title(text):
    print("\n" + "=" * 110)
    print(text)
    print("=" * 110)


def subtitle(text):
    print("\n" + "-" * 110)
    print(text)
    print("-" * 110)


def units_dataframe(result):
    rows = []
    for doc in result.documents:
        for unit in doc.units:
            rows.append(
                {
                    "doc": doc.document_id,
                    "idx": unit.sequence_index,
                    "notation": unit.final_notation,
                    "functions": "+".join(values(unit.functions)),
                    "primary": value(unit.primary_function),
                    "certainty": value(unit.certainty),
                    "emotion": value(unit.emotion_expressed),
                    "intensity": unit.emotion_intensity,
                    "stance": value(unit.stance),
                    "confidence": round(float(unit.confidence), 3),
                    "method": value(unit.method),
                    "needs_review": unit.needs_review,
                    "review_reasons": ", ".join(unit.review_reasons),
                    "heuristics": ", ".join(
                        f"{value(c.label)}:{c.confidence:.2f}" for c in unit.heuristic_candidates
                    ),
                    "text": unit.text,
                }
            )
    return pd.DataFrame(rows)


def print_compact_adn(result):
    title(f"ADN compacto | run_id={result.run_id}")
    for doc in result.documents:
        print(f"\nDOCUMENTO: {doc.document_id}")
        for i, unit in enumerate(doc.units, start=1):
            print(f"ADN{i:02d} {unit.final_notation:16s} -> {unit.text}")


def print_evidence_spans(spans, fallback_text=None, indent="    "):
    if not spans:
        print(f"{indent}evidencia: {fallback_text or '<sin evidence_spans>'}")
        return
    print(f"{indent}evidencia:")
    for j, span in enumerate(spans, start=1):
        bits = []
        if span.char_start is not None and span.char_end is not None:
            bits.append(f"chars={span.char_start}-{span.char_end}")
        if span.start_ms is not None and span.end_ms is not None:
            bits.append(f"ms={span.start_ms}-{span.end_ms}")
        if span.source:
            bits.append(f"source={span.source}")
        suffix = f" ({', '.join(bits)})" if bits else ""
        print(f"{indent}  {j}. {span.text}{suffix}")


def print_unit_detail(unit, idx):
    print(f"\nADN{idx:02d} {unit.final_notation} -> {unit.text}")
    print(f"    unit_id: {unit.unit_id}")
    print(f"    funciones: {values(unit.functions)}")
    print(f"    primaria/secundarias: {value(unit.primary_function)} / {values(unit.secondary_functions)}")
    print(f"    certeza/emoci?n/intensidad/postura: {value(unit.certainty)} / {value(unit.emotion_expressed)} / {unit.emotion_intensity} / {value(unit.stance)}")
    print(f"    confianza/método/revisión: {unit.confidence} / {value(unit.method)} / {unit.needs_review}")
    if unit.review_reasons:
        print("    review_reasons:", ", ".join(unit.review_reasons))
    print_evidence_spans(unit.evidence_spans, fallback_text=unit.text)
    if unit.heuristic_candidates:
        print("    heuristic_candidates:")
        for c in unit.heuristic_candidates:
            print(f"      - {value(c.label)} conf={c.confidence:.2f} | {c.reason}")
    if unit.validator_flags:
        print("    validator_flags:")
        for flag in unit.validator_flags:
            print(f"      - [{value(flag.severity)}] {flag.rule_id}: {flag.message}")
    if unit.rejected_labels:
        print("    rejected_labels:")
        for rejected in unit.rejected_labels:
            print(f"      - {rejected.label} conf={rejected.confidence}: {rejected.reason}")


def print_deep_units(result, limit=None):
    title(f"Detalle por unidad | run_id={result.run_id}")
    count = 0
    for doc in result.documents:
        subtitle(f"Documento {doc.document_id}")
        for i, unit in enumerate(doc.units, start=1):
            if limit is not None and count >= limit:
                print(f"\n... truncado a {limit} unidades")
                return
            print_unit_detail(unit, i)
            count += 1


def print_relations_and_chains(result):
    title(f"Relaciones y cadenas | run_id={result.run_id}")
    for doc in result.documents:
        print(f"\nDOCUMENTO: {doc.document_id}")
        print("relations:", len(doc.relations))
        for rel in doc.relations:
            print(f"  {rel.source_unit_id} --{value(rel.relation_type)}--> {rel.target_unit_id} conf={rel.confidence} evidence={rel.evidence}")
        print("chains:", len(doc.chains))
        for chain in doc.chains:
            print(f"  {chain.chain_id} type={value(chain.chain_type)} units={chain.unit_ids} strength={chain.strength}")


def print_audit_and_timing(run_dir):
    run_dir = Path(run_dir)
    title(f"Archivos de auditor?a | {run_dir}")
    for name in ["run_manifest.json", "audit_report.md", "dna_sequences.txt", "timing_report.json"]:
        path = run_dir / name
        print(f"{name}:", "OK" if path.exists() else "no existe")
    audit_md = run_dir / "audit_report.md"
    if audit_md.exists():
        subtitle("audit_report.md preview")
        print(audit_md.read_text(encoding="utf-8")[:1600])
    timing_path = run_dir / "timing_report.json"
    if timing_path.exists():
        timing = json.loads(timing_path.read_text(encoding="utf-8"))
        subtitle("timing_report.json resumen")
        print(json.dumps(timing.get("summary", timing) if isinstance(timing, dict) else timing[:3], indent=2, ensure_ascii=False)[:2000])


def print_pipeline_result(result, *, deep=False, limit=None):
    print_compact_adn(result)
    subtitle("Tabla resumen")
    display(units_dataframe(result))
    if deep:
        print_deep_units(result, limit=limit)
    print_relations_and_chains(result)
    print_audit_and_timing(result.run_dir)

## 7. Inspeccionar el no-LLM baseline

Fíjate en dos cosas: ya no todo es `N_N0{0}`, y las unidades candidatas/multietiqueta quedan marcadas para revisión.

In [ ]:
print_pipeline_result(no_llm_result, deep=True)

## 8. Leer outputs JSON/JSONL directamente

Este bloque muestra c?mo auditar los archivos generados. Los CSV existen s?lo como export derivado.

In [ ]:
run_dir = Path(no_llm_result.run_dir)
manifest = json.loads((run_dir / "run_manifest.json").read_text(encoding="utf-8"))
units = [json.loads(line) for line in (run_dir / "units.jsonl").read_text(encoding="utf-8").splitlines()]
relations = [json.loads(line) for line in (run_dir / "relations.jsonl").read_text(encoding="utf-8").splitlines()]
chains = [json.loads(line) for line in (run_dir / "chains.jsonl").read_text(encoding="utf-8").splitlines()]

print("manifest keys:", sorted(manifest.keys()))
print("taxonomy_version_effective:", manifest.get("taxonomy_version_effective"))
print("prompt_version_effective:", manifest.get("prompt_version_effective"))
print("validator_version_effective:", manifest.get("validator_version_effective"))
print("units/relations/chains:", len(units), len(relations), len(chains))

print("\nPrimeras unidades JSONL:")
for unit in units[:5]:
    print(json.dumps({
        "unit_id": unit["unit_id"],
        "final_notation": unit["final_notation"],
        "functions": unit["functions"],
        "method": unit["method"],
        "needs_review": unit["needs_review"],
        "heuristic_candidates": unit["heuristic_candidates"],
        "text": unit["text"],
    }, ensure_ascii=False, indent=2))

## 9. Regresión específica: `peque?o` no debe ser pregunta

Este caso reproduce el bug original: un `?` incrustado dentro de una palabra da?ada no debe bloquear `P`. Una pregunta real al final s? debe bloquear `P`.

In [ ]:
broken_result = run_pipeline_from_text(
    broken_question_text,
    document_id="colab_broken_question_regression",
    source_path="<colab-string>",
    output_dir="outputs",
    run_id="colab_broken_question_regression",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
)

print_pipeline_result(broken_result, deep=True)

rows = units_dataframe(broken_result)
print("\nChequeo r?pido:")
print(rows[["idx", "notation", "functions", "text"]])
assert "P" not in rows.iloc[0]["functions"], "peque?o/br?jula disparó P indebidamente"
assert "P" in rows.iloc[-1]["functions"], "la pregunta real no disparó P"
print("OK: texto da?ado no dispara P; pregunta real s? dispara P")

## 10. Ejecutar desde archivo `.txt`

Mismo pipeline, pero leyendo un archivo local. ?til si subes transcripciones a Colab.

In [ ]:
Path("colab_challenging_example.txt").write_text(transcript_text, encoding="utf-8")
txt_result = run_pipeline(
    input_dir="colab_challenging_example.txt",
    output_dir="outputs",
    run_id="colab_txt_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
)
print("run_dir:", txt_result.run_dir)
print_compact_adn(txt_result)

## 11. Alternativa por CLI

La CLI escribe los mismos outputs JSON-first en `outputs/<run_id>/`.

In [ ]:
subprocess.run(
    [
        "narrative-dna",
        "run",
        "--input-dir",
        "colab_challenging_example.txt",
        "--output-dir",
        "outputs",
        "--run-id",
        "colab_cli_txt_no_llm",
        "--no-llm",
        "--no-adjudicator",
    ],
    check=True,
)
subprocess.run(["narrative-dna", "inspect", "--run-id", "colab_cli_txt_no_llm"], check=True)

## 12. Pruebas r?pidas del repo

Estas pruebas cubren golden regression, heur?sticas, cliente LLM y pipeline no-LLM.

In [ ]:
!python -m pytest tests/test_golden_regression.py tests/test_heuristic_candidates.py tests/test_llm_client.py tests/test_pipeline.py -q --basetemp .cache/pytest-colab

## 13. OpenAI opcional: clasificaci?n LLM + adjudicaci?n

Para ejecutar esta secci?n:

1. En Colab, abre **Secrets**.
2. Crea `OPENAI_API_KEY`.
3. Activa acceso al notebook.
4. Ejecuta esta celda.

El cliente usa Responses API con Structured Outputs estrictos. Para GPT-5.5 no env?a `temperature`; usa `reasoning.effort` y `text.verbosity` desde `configs/llm_config.json`.

In [ ]:
try:
    from google.colab import userdata  # type: ignore
    secret_key = userdata.get("OPENAI_API_KEY")
except Exception as exc:
    print("No se pudo leer Colab Secrets:", repr(exc))
    secret_key = None

if not secret_key:
    print("OPENAI_API_KEY no configurada. Se omite la ejecuci?n LLM.")
    llm_result = None
else:
    os.environ["OPENAI_API_KEY"] = secret_key
    llm_result = run_pipeline_from_text(
        transcript_text,
        document_id="colab_challenging_demo_llm",
        source_path="<colab-string>",
        metadata={"source": "colab_challenging_example", "mode": "llm"},
        language="es",
        output_dir="outputs",
        run_id="colab_challenging_llm_demo",
        use_llm=True,
        use_adjudicator=True,
        audit_similarity_enabled=True,
        log_timings=True,
    )
    print("run_dir:", llm_result.run_dir)
    print_pipeline_result(llm_result, deep=True)

## 14. Comparar no-LLM vs LLM

Esta tabla ayuda a ver cu?ndo el LLM agreg? valor y cu?ndo el baseline ya era suficiente. Si `llm_result` es `None`, ejecuta primero la secci?n de OpenAI.

In [ ]:
def compare_results(left, right, left_name="no_llm", right_name="llm"):
    left_df = units_dataframe(left)[["idx", "notation", "functions", "confidence", "method", "needs_review", "text"]]
    left_df = left_df.rename(columns={
        "notation": f"notation_{left_name}",
        "functions": f"functions_{left_name}",
        "confidence": f"confidence_{left_name}",
        "method": f"method_{left_name}",
        "needs_review": f"needs_review_{left_name}",
    })
    right_df = units_dataframe(right)[["idx", "notation", "functions", "confidence", "method", "needs_review"]]
    right_df = right_df.rename(columns={
        "notation": f"notation_{right_name}",
        "functions": f"functions_{right_name}",
        "confidence": f"confidence_{right_name}",
        "method": f"method_{right_name}",
        "needs_review": f"needs_review_{right_name}",
    })
    merged = left_df.merge(right_df, on="idx", how="outer")
    merged["changed"] = merged[f"notation_{left_name}"] != merged[f"notation_{right_name}"]
    return merged

if llm_result is None:
    print("No hay llm_result; configura OPENAI_API_KEY y ejecuta la secci?n anterior.")
else:
    comparison = compare_results(no_llm_result, llm_result)
    display(comparison)
    print("Cambios de notaci?n:", int(comparison["changed"].sum()), "de", len(comparison))

## 15. Ejemplo corto: caja negra emocional

?til para probar un texto más narrativo/personal. Por defecto corre sin LLM para no gastar; cambia `use_llm=True` si ya configuraste Secrets.

In [ ]:
black_box_result = run_pipeline_from_text(
    black_box_memory_text,
    document_id="colab_black_box_memory_no_llm",
    source_path="<colab-string>",
    output_dir="outputs",
    run_id="colab_black_box_memory_no_llm_demo",
    use_llm=False,
    use_adjudicator=False,
    audit_similarity_enabled=False,
)
print_pipeline_result(black_box_result, deep=True)

## 16. Exports derivados

Los CSV se generan para an?lisis tabular, pero no son fuente de verdad. Reconstr?yelos siempre desde JSON/JSONL si algo cambia.

In [ ]:
exports_dir = Path(no_llm_result.run_dir) / "exports"
print("exports_dir:", exports_dir)
for path in sorted(exports_dir.glob("*.csv")):
    print("\n", path.name)
    display(pd.read_csv(path).head())

## 17. Evaluar contra synthetic gold high-confidence

S?lo `synthetic_gold_high_confidence` debe usarse como regresi?n. Esta celda queda comentada porque primero necesitas generar/promover synthetic gold para el run.

In [ ]:
# gold = "outputs/colab_challenging_llm_demo/synthetic_gold_high_confidence.jsonl"
# !narrative-dna evaluate --run-id colab_challenging_llm_demo --gold {gold}

## 18. Checklist de depuraci?n

Si ves resultados pobres:

- Revisa `outputs/<run_id>/units.jsonl`, no el CSV.
- Confirma que `final_notation` se deriva del JSON validado.
- Mira `heuristic_candidates`, `validator_flags`, `review_reasons` y `evidence_spans`.
- En LLM, abre `timing_report.json`: cada llamada OpenAI debe tener `api_call_purpose`.
- Si hay muchos `N_N0{0}`, distingue entre ausencia real de se?ales y clasificador fallando.
- Para GPT-5.5 no debe enviarse `temperature`; la diversidad debe venir del prompt/perfil.